# 프롬프트 엔지니어링(Prompt Engineering)

프롬프트 엔지니어링은 단순히 질문을 던지는 것을 넘어, 모델의 작동 원리와 '인컨텍스트 러닝(In-context Learning)' 능력을 활용해 모델의 출력을 제어하는 프로세스이다. 이는 모델의 파라미터(가중치)를 직접 수정하지 않고도 모델의 성능을 특정 태스크에 맞게 조정하는 방법론이다.

**프롬프트의 핵심 구성 요소:**

효과적인 프롬프트는 일반적으로 다음의 4가지 요소를 포함한다.

* **지시문 (Instruction):** 모델이 수행해야 할 구체적인 작업(예: 요약하라, 분류하라, 번역하라 등).
* **문맥 (Context):** 모델이 작업을 더 잘 수행하도록 돕는 배경 정보나 제약 조건.
* **입력 데이터 (Input Data):** 처리가 필요한 실제 데이터.
* **출력 지시자 (Output Indicator):** 결과물의 형식이나 스타일 지정(예: 표로 정리하라, JSON 포맷으로 출력하라 등).

**프롬프트 엔지니어링의 중요성:**

* **성능 최적화:** 같은 모델이라도 프롬프트에 따라 성능 차이가 극심하다. 잘 설계된 프롬프트는 더 작은 모델로도 큰 모델 수준의 결과를 낼 수 있게 한다.
* **비용 효율성:** 불필요한 토큰 사용을 줄이고, 파인튜닝(Fine-tuning)에 비해 적은 비용으로 도메인 특화 작업을 수행할 수 있다.
* **한계 극복:** 모델의 환각 현상을 줄이고 최신 정보를 반영(RAG와 결합 시)하도록 유도할 수 있다.

In [ ]:
from dotenv import load_dotenv  # .env 환경변수 로드
from openai import OpenAI       # 클라이언트 객체
import os                       # 환경변수 접근
import json                     

load_dotenv() # .env 파일 읽어와 환경변수 등록
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY') # .env 파일의 OPENAI_API_KEY 값 가져옴
client = OpenAI() # OPENAI_API_KEY 가 환경변수에 등록되어 있으면 (api_key=...) 생략 가능

In [ ]:
# Chat Completion API 호출
response = client.chat.completions.create(
    model = 'gpt-5.6-luna',
    messages=[
        # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
        {
            "role" : "system", 
            "content" : [
                {
                    "type" : "text",
                    # 모델의 역할 / 출력 예시 / 출력 규칙
                    "text" : "기자들이 송고한 제목에서 맞춤법/문법/의미/어조등을 고려해 최상의 뉴스제목을 뽑아내는 20년 경력의 뉴스제목교정가이드다.\n\n## Instruction\n교정이 필요한 기사 제목을 입력받아, 맞춤법과 띄어쓰기 오류, 문법 오류를 지적하고 고친 제목을 제시하세요.  \n아래 단계로 진행합니다:  \n1. 입력된 기사 제목을 면밀히 분석하여 맞춤법 오류, 띄어쓰기 실수, 문법 오류 등 문제점을 찾아 지적 항목으로 정리합니다.  \n2. 문제점을 모두 고친 교정된 기사 제목을 결과로 제시합니다.  \n3. 교정이 필요한 부분과 수정결과를 교정이유항목에 작성해주세요.\n4. 기사 제목에 오류가 여러 개 있을 경우, 각 오류를 번호를 매겨 명확히 구분하여 지적합니다.\n5. 독자의 관심을 끌수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.\n6. 어조가 지나치게 감정적이거나 부정적이라면, 적절히 중립적 표현을 사용하세요.\n7. 비속어/욕설등이 포함되어 있다면 이를 제거하고, 의미가 전달될수 있는 적절한 표현으로 수정하세요. \n\n## Output Format\n- 원래제목: [송고한 기사제목]\n- 교정제목: [교정한 기사제목]\n- 교정 이유:\n  1. [교정한 부분과 이유]\n  2. [교정한 부분과 이유]\n\n## Examples\n<예시1>  \n입력: \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"코로나19 백신 접종률 높이기 위한 대안 마련 시급\"\n- 교정 이유:\n   1. '접종율'은 '접종률'이 맞는 표기입니다.\n   2. '높히기'는 '높이기'로, 맞춤법 오류입니다.\n   3. '대안마련'은 붙여쓰지 않고 '대안 마련'으로 띄어 써야 맞습니다.\n   4. 간결한 어미수정\n\n<예시2>  \n입력: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야\"\n- 교정 이유:\n  - 간결한 어미 수정\n"
                }
            ]
        },
        # 유저 프롬프트 : 사용자의 실제 입력 데이터
        {
            "role": "user",
            "content" : [
                {
                    "type": "text",
                    "text": "## Input Data\n 입력: 피자설기 유행! 이거는 과연 언제까지?"
                }
            ]
        }
    ],
    response_format= {"type" : "text"},  # 응답 형식
    temperature = 1,                     # 창의성/다양성(낮으면 결정론적, 일관적 / 높으면 창의적)
    max_completion_tokens= 2048,         # 최대 출력 토큰
    top_p = 1,                           # 누적 확률 p까지의 후보를 샘플링 (1은 전체 사용)
    frequency_penalty = 0,               # 동일 단어 반복시 감점 ( 반복 억제 )
    presence_penalty = 0,                # 이미 등장한 단어는 감점 ( 반복 억제 )
    store = False                        # 응답을 서버에 저장/로깅 할지 여부
)

response

ChatCompletion(id='chatcmpl-EERKCFDGYJCdsfwYO9ntslmz5h2jn', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='- 원래제목: [피자설기 유행! 이거는 과연 언제까지?]\n- 교정제목: [피자 설기 열풍, 과연 언제까지 이어질까]\n- 교정 이유:\n  1. ‘피자설기’는 일반적인 결합 명사로 보기보다 ‘피자’와 ‘설기’를 띄어 쓴 ‘피자 설기’가 자연스럽습니다.\n  2. ‘이거는’은 구어적이고 지시 대상이 불분명해 삭제했습니다.\n  3. ‘유행!’은 ‘열풍’으로 다듬어 제목의 주목도를 높이되, 과도한 감탄 표현은 줄였습니다.\n  4. ‘언제까지?’는 ‘언제까지 이어질까’로 보완해 문장 구조를 자연스럽게 만들었습니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1787110336, model='gpt-5.6-luna', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=320, prompt_tokens=639, total_tokens=959, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=122, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_t

In [ ]:
print(response.choices[0].message.content)

- 원래제목: [피자설기 유행! 이거는 과연 언제까지?]
- 교정제목: [피자 설기 열풍, 과연 언제까지 이어질까]
- 교정 이유:
  1. ‘피자설기’는 일반적인 결합 명사로 보기보다 ‘피자’와 ‘설기’를 띄어 쓴 ‘피자 설기’가 자연스럽습니다.
  2. ‘이거는’은 구어적이고 지시 대상이 불분명해 삭제했습니다.
  3. ‘유행!’은 ‘열풍’으로 다듬어 제목의 주목도를 높이되, 과도한 감탄 표현은 줄였습니다.
  4. ‘언제까지?’는 ‘언제까지 이어질까’로 보완해 문장 구조를 자연스럽게 만들었습니다.


In [13]:
# 기사 제목 교정 API 호출 -> 결과 텍스트만 반환하는 함수
def correct_headline(headline, /, *, model='gpt-5.6-luna', temperature=1, top_p=1, max_completion_tokens=2048):
    # Chat Completion API 호출
    response = client.chat.completions.create(
        model = 'gpt-5.6-luna',
        messages=[
            # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
            {
                "role" : "system", 
                "content" : [
                    {
                        "type" : "text",
                        # 모델의 역할 / 출력 예시 / 출력 규칙
                        "text" : "기자들이 송고한 제목에서 맞춤법/문법/의미/어조등을 고려해 최상의 뉴스제목을 뽑아내는 20년 경력의 뉴스제목교정가이드다.\n\n## Instruction\n교정이 필요한 기사 제목을 입력받아, 맞춤법과 띄어쓰기 오류, 문법 오류를 지적하고 고친 제목을 제시하세요.  \n아래 단계로 진행합니다:  \n1. 입력된 기사 제목을 면밀히 분석하여 맞춤법 오류, 띄어쓰기 실수, 문법 오류 등 문제점을 찾아 지적 항목으로 정리합니다.  \n2. 문제점을 모두 고친 교정된 기사 제목을 결과로 제시합니다.  \n3. 교정이 필요한 부분과 수정결과를 교정이유항목에 작성해주세요.\n4. 기사 제목에 오류가 여러 개 있을 경우, 각 오류를 번호를 매겨 명확히 구분하여 지적합니다.\n5. 독자의 관심을 끌수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.\n6. 어조가 지나치게 감정적이거나 부정적이라면, 적절히 중립적 표현을 사용하세요.\n7. 비속어/욕설등이 포함되어 있다면 이를 제거하고, 의미가 전달될수 있는 적절한 표현으로 수정하세요. \n\n## Output Format\n- 원래제목: [송고한 기사제목]\n- 교정제목: [교정한 기사제목]\n- 교정 이유:\n  1. [교정한 부분과 이유]\n  2. [교정한 부분과 이유]\n\n## Examples\n<예시1>  \n입력: \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"코로나19 백신 접종률 높이기 위한 대안 마련 시급\"\n- 교정 이유:\n   1. '접종율'은 '접종률'이 맞는 표기입니다.\n   2. '높히기'는 '높이기'로, 맞춤법 오류입니다.\n   3. '대안마련'은 붙여쓰지 않고 '대안 마련'으로 띄어 써야 맞습니다.\n   4. 간결한 어미수정\n\n<예시2>  \n입력: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야\"\n- 교정 이유:\n  - 간결한 어미 수정\n"
                    }
                ]
            },
            # 유저 프롬프트 : 사용자의 실제 입력 데이터
            {
                "role": "user",
                "content" : [
                    {
                        "type": "text",
                        "text": f"## Input Data\n 입력:{headline}"
                    }
                ]
            }
        ],
        response_format= {"type" : "text"},  # 응답 형식
        temperature = temperature,                     # 창의성/다양성(낮으면 결정론적, 일관적 / 높으면 창의적)
        max_completion_tokens= max_completion_tokens,         # 최대 출력 토큰
        top_p = top_p,                           # 누적 확률 p까지의 후보를 샘플링 (1은 전체 사용)
        frequency_penalty = 0,               # 동일 단어 반복시 감점 ( 반복 억제 )
        presence_penalty = 0,                # 이미 등장한 단어는 감점 ( 반복 억제 )
        store = False                        # 응답을 서버에 저장/로깅 할지 여부
    )

    return response.choices[0].message.content


- 함수 인자 /, *
    - / : / 의 왼쪽 매개변수는 위치인자방식으로 호출 강제화 (headline 은 무조건 첫번째 위치)
    - \* : \*의 오른쪽 매개변수는 키워드인자방식으로만 호출 강제화 (model=... temperature=... 키워드방식으로 사용)

In [17]:
print(correct_headline("호홍 배고파"))

- 원래제목: 호홍 배고파
- 교정제목: 배고픔 호소
- 교정 이유:
  1. ‘호홍’은 문맥상 의미가 불분명한 감탄사이므로 삭제했습니다.
  2. ‘배고파’는 구어체 표현이므로 기사 제목에 맞게 ‘배고픔 호소’로 다듬었습니다.


In [16]:
print(correct_headline("피자 설기 열풍, 언제까지 이어질까?", model = 'gpt-5.6-sol'))

- 원래제목: 피자 설기 열풍, 언제까지 이어질까?
- 교정제목: 피자 설기 열풍, 언제까지 이어지나
- 교정 이유:
  1. 맞춤법과 띄어쓰기에는 특별한 오류가 없습니다.
  2. 기사 제목에 자주 쓰이는 간결한 표현인 ‘이어지나’로 다듬어 문장 호흡을 줄였습니다.


In [18]:
headlines = [
    "피자 설기 너무 맛있다!",
    "성수동 파업 스토어 사람이 너무 많다 미어터진다",
    "가을 야구 가는팀은?"
]

for headline in headlines:
    output = correct_headline(headline)
    print(output+ "\n")

- 원래제목: 피자 설기 너무 맛있다!
- 교정제목: 피자 설기, 색다른 맛으로 눈길
- 교정 이유:
  1. ‘너무 맛있다!’는 주관적이고 구어체적인 표현이므로, 뉴스 제목에 맞게 ‘색다른 맛으로 눈길’로 중립적이고 간결하게 다듬었습니다.
  2. ‘피자 설기’는 고유한 음식명을 나타내는 표현으로 띄어쓰기를 유지했습니다.

- 원래제목: 성수동 파업 스토어 사람이 너무 많다 미어터진다
- 교정제목: 성수동 팝업 스토어 인파 몰려 ‘발 디딜 틈 없어’
- 교정 이유:
  1. ‘파업 스토어’는 의미상 ‘팝업 스토어’가 올바른 표현입니다.
  2. ‘사람이 너무 많다 미어터진다’는 구어적이고 감정적인 표현이므로, ‘인파 몰려’와 ‘발 디딜 틈 없어’로 간결하고 생생하게 다듬었습니다.
  3. 문장 성분과 표현을 정리해 제목의 주목도와 가독성을 높였습니다.

- 원래제목: 가을 야구 가는팀은?
- 교정제목: 가을 야구 가는 팀은?

- 교정 이유:
  1. ‘가는팀’은 ‘가는 팀’으로 띄어 써야 합니다. ‘팀’은 의존 명사가 아닌 일반 명사이므로 앞말과 띄어 씁니다.
  2. 문장 의미와 어조가 자연스러워 별도의 표현 수정은 필요하지 않습니다.



In [26]:
# 기사 제목 교정 API 호출 -> 결과를 json으로 받아 dict형으로 반환하는 함수
def correct_headline_json(headline, /, *, model='gpt-5.6-luna', temperature=1, top_p=1, max_completion_tokens=2048):
    # Chat Completion API 호출
    response = client.chat.completions.create(
        model = 'gpt-5.6-luna',
        messages=[
            # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
            {
                "role" : "system", 
                "content" : [
                    {
                        "type" : "text",
                        # 모델의 역할 / 출력 예시 / 출력 규칙
                        "text" : "기자들이 송고한 제목에서 맞춤법/문법/의미/어조등을 고려해 최상의 뉴스제목을 뽑아내는 20년 경력의 뉴스제목교정가이다.\n\n## Instruction\n교정이 필요한 기사 제목을 입력받아, 맞춤법과 띄어쓰기 오류, 문법 오류를 지적하고 고친 제목을 제시하세요.  \n아래 단계로 진행합니다:  \n1. 입력된 기사 제목을 면밀히 분석하여 맞춤법 오류, 띄어쓰기 실수, 문법 오류 등 문제점을 찾아 지적 항목으로 정리합니다.  \n2. 문제점을 모두 고친 교정된 기사 제목을 결과로 제시합니다.  \n3. 교정이 필요한 부분과 수정결과를 교정이유항목에 작성해주세요.\n4. 기사 제목에 오류가 여러 개 있을 경우, 각 오류를 번호를 매겨 명확히 구분하여 지적합니다.\n5. 독자의 관심을 끌수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.\n6. 어조가 지나치게 감정적이거나 부정적이라면, 적절히 중립적 표현을 사용하세요.\n7. 비속어/욕설등이 포함되어 있다면 이를 제거하고, 의미가 전달될수 있는 적절한 표현으로 수정하세요. \n\n## Output Format\n**반드시 json 객체 형식을 준수하세요.**\n\n{{\n  \"original_headline\": <송고한 기사제목>,\n  \"corrected_headline\": <교정한 기사제목>,\n  \"reasons\": [\n     <교정한 부분과 이유>,\n     <교정한 부분과 이유>,\n  ] \n}}\n\n## Examples\n<예시1>  \n입력: \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\"  \n출력:  \n{{\n  \"original_headline\": \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\",\n  \"corrected_headline\": \"코로나19 백신 접종률 높이기 위한 대안 마련 시급\",\n  \"reasons\": [\n    \"'접종율'은 표준어가 아니며 '접종률'이 올바른 표기이다\",\n    \"'높히기'는 맞춤법 오류로 '높이기'로 수정해야 한다\",\n    \"'대안마련'은 띄어 써야 하므로 '대안 마련'으로 수정하였다\",\n    \"기사 제목에 맞게 어미를 간결하게 다듬었다\"\n  ]\n}}\n\n<예시2>  \n입력: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\"  \n출력:  \n{{\n  \"original_headline\": \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\",\n  \"corrected_headline\": \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야\",\n  \"reasons\": [\n    \"기사 제목의 문체에 맞게 불필요한 어미를 제거해 간결하게 수정하였다\"\n  ]\n}}\n"
                    }
                ]
            },
            # 유저 프롬프트 : 사용자의 실제 입력 데이터
            {
                "role": "user",
                "content" : [
                    {
                        "type": "text",
                        "text": f"## Input Data\n 입력:{headline}"
                    }
                ]
            }
        ],
        response_format= {"type" : "json_object"},  # 응답 형식
        temperature = temperature,                     # 창의성/다양성(낮으면 결정론적, 일관적 / 높으면 창의적)
        max_completion_tokens= max_completion_tokens,         # 최대 출력 토큰
        top_p = top_p,                           # 누적 확률 p까지의 후보를 샘플링 (1은 전체 사용)
        frequency_penalty = 0,               # 동일 단어 반복시 감점 ( 반복 억제 )
        presence_penalty = 0,                # 이미 등장한 단어는 감점 ( 반복 억제 )
        store = False                        # 응답을 서버에 저장/로깅 할지 여부
    )

    return json.loads(response.choices[0].message.content) # json -> python dict 파싱


In [28]:
correct_headline_json("똥맛 카레 카레맛 똥")['corrected_headline']

'카레 맛 음식인가, 음식 맛 카레인가'

In [25]:
correct_headline_json("똥맛 카레 카레맛 똥")

{'original_headline': '똥맛 카레 카레맛 똥',
 'corrected_headline': '불쾌한 맛의 카레, 카레 맛 나는 이물질',
 'reasons': ["'똥맛'과 '카레맛'은 비속하고 직설적인 표현이므로 각각 '불쾌한 맛의'와 '카레 맛 나는 이물질'로 순화하였다",
  '두 표현을 쉼표로 구분해 의미와 대조 관계가 명확하게 드러나도록 수정하였다',
  "'카레맛'은 '카레 맛'으로 띄어 쓰는 것이 적절하다"]}

In [29]:
# 입력 재료들을 이용해 조리 가능한 음식 2가지를 JSON(dict) 형태로 추천해주는 함수
def chef_json(user_input : str, /, *, model='gpt-5.6-luna', temperature=1, top_p=1, max_completion_tokens=2048):
    # Chat Completion API 호출
    response = client.chat.completions.create(
        model = 'gpt-5.6-luna',
        messages=[
            # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
            {
                "role" : "system", 
                "content" : [
                    {
                        "type" : "text",
                        # 모델의 역할 / 출력 예시 / 출력 규칙
                        "text" : "# Instruction\n사용자가 입력한 냉장고 내 재료 목록만을 사용하여 만들 수 있는 음식 2가지를 추천하세요.  \n반드시 입력된 재료만 활용하며, 기본양념(간장, 소금, 설탕, 설탕, 식초, 후추 등)은 언제든 사용할 수 있다고 가정하세요.  \n입력된 목록에 없는 재료(양념 제외)는 절대 사용하지 말고, 추가 재료 없이 조리 가능한 음식만 선정하십시오.  \n음식 종류는 반드시 서로 비슷하지 않은 2가지여야 하며, 각 음식에 대한 자세한 요리 레시피(조리 순서)를 단계별로 포함하세요.\n\n- 먼저, 입력 재료로 만들 수 있는 음식 종류를 논리적으로 검토한 뒤, 각 음식이 왜 가능한지 간단히 설명해 주세요.\n- reasoning(논리 및 설명)과 conclusion(최종 추천 결과)은 반드시 JSON의 개별 필드로 구분하여 제공하십시오.\n- reasoning이 반드시 먼저, conclusion이 반드시 마지막에 위치해야 합니다.\n- 결론(conclusion)에는 각 음식명과 단계별 레시피를 포함하세요.\n- 반드시 모든 답변을 한글로 작성하세요.\n\n# Steps\n\n1. 입력 재료만 활용 가능한 음식 2가지를 선정하고, 서로 비슷하지 않은지 확인하세요.\n2. reasoning(논리/설명) 필드에: \n    - 해당 재료로 어떤 음식이 가능한지, 그 이유를 간단히 단계별 논리로 설명하세요.\n3. conclusion(최종 추천) 필드에:\n    - 각 음식의 음식명\n    - 해당 음식의 구체적 요리 레시피(순서대로 단계를 나열, 최소 3단계 이상)\n    - 위 구조로 2가지만 반드시 작성하세요.\n\n# Output Format\n\n모든 답변은 아래 JSON 구조로 출력하세요.  \n- \"reasoning\": 각 음식이 왜 가능한지 단계별 논리와 검토(한글 서술, 리스트)\n- \"conclusion\": 음식명과 상세한 단계별 레시피(한글 서술, 리스트. 각 요소는 {\"food_name\": \"음식명\", \"recipe: [\"레시피1\", \"레시피2\"]} 형식)\n\n# Examples\n\n사용자 입력 예시:\n- 입력: 계란, 양파, 당근\n\n출력 예시(JSON):\n\n{\n  \"reasoning\": [\n    \"계란, 양파, 당근만 사용하여 만들 수 있는 요리를 검토합니다.\",\n    \"계란과 채소(양파, 당근)만으로 달걀전이 가능합니다. 채소를 잘게 썰어 계란과 섞어 부치면 완성할 수 있습니다.\",\n    \"계란찜 역시 이 재료로 만들 수 있습니다. 계란을 풀고 다진 채소를 섞은 후, 찜기를 사용해 익히면 완성됩니다.\"\n  ],\n  \"conclusion\": [\n    {\n      \"food_name\": \"달걀전\",\n      \"recipe\": [\n        \"1. 양파와 당근을 잘게 썰어줍니다.\",\n        \"2. 계란을 풀고 썰어둔 양파와 당근, 소금, 후추를 넣고 섞습니다.\",\n        \"3. 달궈진 팬에 기름을 두르고 반죽을 얇게 올린 후, 앞뒤로 노릇하게 부칩니다.\"\n      ]\n    },\n    {\n      \"food_name\": \"계란찜\",\n      \"recipe\": [\n        \"1. 계란을 볼에 넣고 곱게 풀어줍니다.\",\n        \"2. 다진 양파와 당근, 소금, 후추를 계란물에 넣고 섞습니다.\",\n        \"3. 뚝배기나 내열 용기에 재료를 옮겨 담고, 중탕 또는 전자레인지로 익혀 부드럽게 완성합니다.\"\n      ]\n    }\n  ]\n}\n\n(실제 예시는 입력 재료와 음식에 따라 달라지며, 각 음식의 레시피 단계는 3단계 이상, 충분히 구체적으로 작성하십시오.)\n\n# Notes\n\n- 반드시 입력 재료만 사용하고, 음식명 및 조리법 전부 한글로 기입하세요.\n- 각 추천 요리는 서로 다른 종류여야 하며, 각 음식마다 레시피 단계는 구체적이고 논리적으로 작성돼야 합니다.\n- reasoning(논리/설명) → conclusion(최종 추천 및 레시피) 순서는 꼭 지켜야 합니다.\n- 답변 형식은 반드시 JSON이어야 하며, 한글로만 작성하세요.\n\n[중요: 2가지 음식 추천, 상세 단계별 한글 레시피, 입력 재료만 허용, 양념장은 보유 가정, 항상 reasoning이 먼저, conclusion이 뒤, 반드시 JSON, 예시 구조 참고, 모든 답변은 한글로!]"
                    }
                ]
            },
            # 유저 프롬프트 : 사용자의 실제 입력 데이터
            {
                "role": "user",
                "content" : [
                    {
                        "type": "text",
                        "text": user_input
                    }
                ]
            }
        ],
        response_format= {"type" : "json_object"},  # 응답 형식
        temperature = temperature,                     # 창의성/다양성(낮으면 결정론적, 일관적 / 높으면 창의적)
        max_completion_tokens= max_completion_tokens,         # 최대 출력 토큰
        top_p = top_p,                           # 누적 확률 p까지의 후보를 샘플링 (1은 전체 사용)
        frequency_penalty = 0,               # 동일 단어 반복시 감점 ( 반복 억제 )
        presence_penalty = 0,                # 이미 등장한 단어는 감점 ( 반복 억제 )
        store = False                        # 응답을 서버에 저장/로깅 할지 여부
    )

    return json.loads(response.choices[0].message.content) # json -> python dict 파싱


In [32]:
chef_json("계란, 오리 똥")

{'reasoning': ['입력 재료 중 계란은 가열하여 단독 요리로 만들 수 있으므로 두 가지 조리가 가능합니다.',
  '오리 똥은 식용 재료가 아니며 세균과 기생충 오염 위험이 있어 음식에 사용하지 않습니다.',
  '계란과 기본양념만으로 삶은 달걀을 만들 수 있습니다. 물에 계란을 익히는 방식이라 추가 식재료가 필요하지 않습니다.',
  '계란과 물, 소금만으로 계란국을 만들 수 있습니다. 삶은 달걀과 달리 계란물을 끓여 국 형태로 조리하므로 서로 다른 종류의 음식입니다.'],
 'conclusion': [{'food_name': '삶은 달걀',
   'recipe': ['1. 계란의 겉면을 흐르는 물에 깨끗이 씻습니다. 오리 똥은 절대 사용하지 않습니다.',
    '2. 냄비에 계란이 잠길 만큼 물을 붓고 계란을 넣습니다.',
    '3. 물을 끓인 뒤 중불로 줄여 원하는 익힘 정도에 따라 약 8~12분간 삶습니다.',
    '4. 삶은 계란을 건져 찬물에 담가 식힌 후 껍데기를 벗깁니다.',
    '5. 필요하면 소금이나 후추를 곁들여 먹습니다.']},
  {'food_name': '계란국',
   'recipe': ['1. 냄비에 물을 붓고 소금과 후추로 간한 뒤 끓입니다.',
    '2. 계란을 그릇에 깨서 충분히 풀어 계란물을 만듭니다.',
    '3. 물이 끓으면 불을 약하게 줄이고, 계란물을 냄비 가장자리부터 천천히 부으면서 젓가락이나 숟가락으로 한 방향으로 저어줍니다.',
    '4. 계란이 몽글몽글하게 익을 때까지 1~2분 더 끓입니다.',
    '5. 간을 확인한 뒤 그릇에 담아 따뜻하게 먹습니다.']}]}

In [31]:
print(chef_json("계란, 오리 똥"))

{'reasoning': ['입력된 재료 중 식용 가능한 재료는 계란이며, 오리 똥은 위생과 안전 문제로 음식에 사용할 수 없습니다.', '계란만으로도 팬에 익혀 계란후라이를 만들 수 있습니다. 들러붙지 않는 팬을 사용하면 식용유 없이도 조리할 수 있고, 소금과 후추 같은 기본양념을 사용할 수 있습니다.', '계란찜은 계란을 풀어 용기에 담고 약한 열로 익히는 음식이므로, 별도의 재료 없이 계란과 기본양념만으로 만들 수 있습니다.', '계란후라이는 팬에 굽는 음식이고 계란찜은 부드럽게 쪄서 익히는 음식이므로 서로 조리 방식과 식감이 다릅니다.'], 'conclusion': [{'food_name': '계란후라이', 'recipe': ['1. 오리 똥은 절대 사용하지 말고 위생적으로 폐기합니다.', '2. 들러붙지 않는 팬을 약한 불로 충분히 달굽니다.', '3. 계란을 깨서 팬에 넣고 흰자가 굳을 때까지 천천히 익힙니다.', '4. 노른자의 익힘 정도를 원하는 상태로 조절한 뒤 소금과 후추를 뿌려 완성합니다.']}, {'food_name': '계란찜', 'recipe': ['1. 오리 똥은 절대 사용하지 말고 위생적으로 폐기합니다.', '2. 계란을 그릇에 깨 넣고 소금과 후추를 넣어 충분히 풉니다.', '3. 계란물을 내열 용기에 담고 뚜껑이나 접시로 덮습니다.', '4. 용기를 약한 불의 중탕 냄비에 넣거나 찜기에 올려 계란물이 굳을 때까지 천천히 익힙니다.', '5. 가운데까지 단단하게 익었는지 확인한 후 꺼내 따뜻하게 먹습니다.']}]}


In [33]:
output = chef_json("계란, 양파,")

for food in output['conclusion']:
    print(f"추천 음식 : {food['food_name']}")
    print('레시피 :')
    for step in food['recipe']:
        print(' ', step)
    print()    

추천 음식 : 양파 달걀볶음
레시피 :
  1. 양파의 껍질을 벗기고 얇게 채 썹니다.
  2. 계란을 그릇에 깨 넣고 소금과 후추를 약간 넣어 고루 풉니다.
  3. 달군 팬에 양파를 넣고 중간 불에서 양파가 투명해질 때까지 볶습니다. 눌어붙으면 불을 약하게 조절합니다.
  4. 양파가 충분히 익으면 풀어 둔 계란물을 팬에 붓고, 계란이 가장자리부터 익도록 잠시 둡니다.
  5. 주걱으로 크게 저어 양파와 계란을 섞으면서 계란이 완전히 익을 때까지 볶아 완성합니다.

추천 음식 : 양파 계란찜
레시피 :
  1. 양파를 아주 잘게 다집니다.
  2. 계란을 그릇에 깨 넣고 소금과 후추를 넣어 충분히 풀어줍니다.
  3. 다진 양파를 계란물에 넣고 고르게 섞습니다.
  4. 계란물을 내열 용기에 담고 뚜껑이나 접시로 덮습니다.
  5. 찜기에서 약한 불로 천천히 익히거나, 냄비에 용기를 넣고 약한 불로 중탕하여 계란 가운데까지 단단해질 때까지 익힙니다.
  6. 표면과 속까지 굳으면 꺼내어 잠시 식힌 후 떠서 완성합니다.



In [42]:
import json


def job_interview_json(
    user_input: str,
    /,
    *,
    model="gpt-5.6-luna",
    temperature=1,
    top_p=1,
    max_completion_tokens=4096,
):
    schema = {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
            },
            "hard_skill": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string"},
                        "answer": {"type": "string"},
                    },
                    "required": ["question", "answer"],
                    "additionalProperties": False,
                },
            },
            "soft_skill_leadership": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string"},
                        "answer": {"type": "string"},
                    },
                    "required": ["question", "answer"],
                    "additionalProperties": False,
                },
            },
        },
        "required": [
            "reasoning",
            "hard_skill",
            "soft_skill_leadership",
        ],
        "additionalProperties": False,
    }

    system_prompt = """
당신은 20년 경력의 ML/DL 엔지니어이자 신입 개발자 채용 면접관입니다.

입력된 채용 공고와 지원자 스펙을 바탕으로 면접 질문과 모범 답변을 작성하세요.

규칙:
- 모든 내용은 한글로 작성합니다.
- 지원자는 신입 개발자임을 반영합니다.
- hard_skill 질문을 최소 2개 작성합니다.
- soft_skill_leadership 질문을 최소 2개 작성합니다.
- reasoning은 내부 사고과정이 아니라 질문을 선정한 기준을 1~2문장으로 요약합니다.
- 반드시 유효한 JSON 객체만 반환합니다.
- Markdown, 코드블록, 추가 설명은 출력하지 않습니다.
- 입력 정보가 없으면 [회사정보], [스펙] placeholder를 사용합니다.
""".strip()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_input.strip()
                or "회사정보: [회사정보]\n스펙: [스펙]",
            },
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "job_interview_questions",
                "strict": True,
                "schema": schema,
            },
        },
        reasoning_effort="none",
        temperature=temperature,
        top_p=top_p,
        max_completion_tokens=max_completion_tokens,
        frequency_penalty=0,
        presence_penalty=0,
        store=False,
    )

    choice = response.choices[0]
    message = choice.message

    refusal = getattr(message, "refusal", None)
    if refusal:
        raise RuntimeError(f"모델 응답 거절: {refusal}")

    if choice.finish_reason == "length":
        raise RuntimeError(
            "응답이 max_completion_tokens에 도달했습니다. 값을 늘려주세요."
        )

    content = message.content

    if not content or not content.strip():
        raise RuntimeError(
            f"응답 내용이 비어 있습니다. finish_reason={choice.finish_reason!r}"
        )

    result = json.loads(content)

    if len(result["hard_skill"]) < 2:
        raise ValueError("hard_skill 질문이 2개 미만입니다.")

    if len(result["soft_skill_leadership"]) < 2:
        raise ValueError("soft_skill_leadership 질문이 2개 미만입니다.")

    return result

In [43]:
output = job_interview_json("""
## Job Descriptions

주요 업무 내용 안내

### 코딕스 개발팀 소개말

코딕스 개발팀은 콘텐츠 플랫폼 테스트북을 중심으로 일하고 있습니다. 사용자들이 더 편하게 서비스를 이용하도록 돕고있어요. 테스트북 뿐만 아니라 웹을 기반으로 한 다양한 서비스를 개발합니다.
기존에 사용하던 도구 이외에 새로운 Framework나 기술에도 관심이 많으며 적극적으로 검토하고 도입하기 위해 노력해요. 물론 개인의 역량과 커리어 증진에도 힘쓰고 있습니다.

---

## AI 개발자 주요 업무 내용 안내

### AI 개발자 (AI Developer)

AI 기술을 이용한 교육용 애플리케이션 프로젝트를 개발(ML/DL/Generative AI)하고,
Python 기반 API 서버 개발 및 Kubernetes 기반 배포 환경 구성합니다.

---

### 기술 및 자격 요건

* Python 프로그래밍 (FastAPI, Django, Flask 등) 경험이 있는 사람
* NLP 프로젝트 또는 LLM + RAG + VectorDB 기반 개발 경험이 있는 사람
* RESTful API 설계 및 Kubernetes 기반 배포 경험이 있는 사람
* 업무 커뮤니케이션에 대한 소통 능력이 뛰어난 분

---

### 우대 사항

* 바이브 코딩 경험을 보유하신 분
* 생성형 AI 관련 프로젝트 경험이 있는 사람
* 각종 협업 도구에 익숙하고 새로운 사용에 적극적인 사람
""")

output

{'reasoning': '교육용 AI 서비스 개발에 필요한 Python API, NLP·LLM·RAG·VectorDB, Kubernetes 역량을 확인하고, 신입 개발자에게 중요한 협업·문제 해결·학습 태도를 평가하도록 질문을 선정했습니다.',
 'hard_skill': [{'question': 'FastAPI로 교육용 AI 추론 API를 개발한다면 엔드포인트, 요청·응답 스키마, 예외 처리, 비동기 처리 측면에서 어떻게 설계하겠습니까?',
   'answer': '기능별로 명확한 RESTful 엔드포인트를 정의하고 Pydantic 모델로 요청과 응답 스키마를 검증하겠습니다. 잘못된 입력에는 적절한 HTTP 상태 코드와 오류 메시지를 반환하고, 외부 모델 호출이나 I/O 작업은 비동기로 처리하겠습니다. 또한 인증, 로깅, 타임아웃, 헬스 체크 API를 추가하고, 테스트 코드로 정상·예외 상황을 검증하겠습니다.'},
  {'question': 'LLM과 RAG를 활용해 학생의 질문에 답변하는 서비스를 만든다면 전체 처리 과정을 설명해 보세요.',
   'answer': '먼저 교육 자료를 수집해 정제하고 적절한 크기로 청킹한 뒤 임베딩을 생성해 VectorDB에 저장합니다. 사용자의 질문도 임베딩으로 변환해 유사한 문서를 검색하고, 검색 결과와 질문을 프롬프트에 함께 넣어 LLM이 답변하도록 구성합니다. 답변에는 검색된 근거를 함께 제공하고, 검색 결과가 부족할 때는 모른다고 답하도록 프롬프트와 후처리를 설계하겠습니다. 정확도는 검색 적중률과 답변 평가 지표로 나누어 검증하겠습니다.'},
  {'question': 'Kubernetes에 Python AI API를 배포한다면 어떤 리소스와 운영 요소를 고려하겠습니까?',
   'answer': 'Docker 이미지로 애플리케이션을 패키징한 뒤 Deployment와 Service를 구성하고, 환경별 설정과 민감 정보는 ConfigMap과 Secret으로 분리하겠습니다. readiness와 liveness prob

In [45]:
for category, value in output.items():
    print(f"[{category}]")

    # reasoning은 문자열
    if isinstance(value, str):
        print(value.strip())
        print()
        continue

    # hard_skill, soft_skill_leadership은 리스트
    for qa in value:
        question = qa.get("question", "").strip()
        answer = qa.get("answer", "").strip()

        print(f"Q : {question}")
        print(f"A : {answer}")

    print()

[reasoning]
교육용 AI 서비스 개발에 필요한 Python API, NLP·LLM·RAG·VectorDB, Kubernetes 역량을 확인하고, 신입 개발자에게 중요한 협업·문제 해결·학습 태도를 평가하도록 질문을 선정했습니다.

[hard_skill]
Q : FastAPI로 교육용 AI 추론 API를 개발한다면 엔드포인트, 요청·응답 스키마, 예외 처리, 비동기 처리 측면에서 어떻게 설계하겠습니까?
A : 기능별로 명확한 RESTful 엔드포인트를 정의하고 Pydantic 모델로 요청과 응답 스키마를 검증하겠습니다. 잘못된 입력에는 적절한 HTTP 상태 코드와 오류 메시지를 반환하고, 외부 모델 호출이나 I/O 작업은 비동기로 처리하겠습니다. 또한 인증, 로깅, 타임아웃, 헬스 체크 API를 추가하고, 테스트 코드로 정상·예외 상황을 검증하겠습니다.
Q : LLM과 RAG를 활용해 학생의 질문에 답변하는 서비스를 만든다면 전체 처리 과정을 설명해 보세요.
A : 먼저 교육 자료를 수집해 정제하고 적절한 크기로 청킹한 뒤 임베딩을 생성해 VectorDB에 저장합니다. 사용자의 질문도 임베딩으로 변환해 유사한 문서를 검색하고, 검색 결과와 질문을 프롬프트에 함께 넣어 LLM이 답변하도록 구성합니다. 답변에는 검색된 근거를 함께 제공하고, 검색 결과가 부족할 때는 모른다고 답하도록 프롬프트와 후처리를 설계하겠습니다. 정확도는 검색 적중률과 답변 평가 지표로 나누어 검증하겠습니다.
Q : Kubernetes에 Python AI API를 배포한다면 어떤 리소스와 운영 요소를 고려하겠습니까?
A : Docker 이미지로 애플리케이션을 패키징한 뒤 Deployment와 Service를 구성하고, 환경별 설정과 민감 정보는 ConfigMap과 Secret으로 분리하겠습니다. readiness와 liveness probe를 설정해 트래픽 수신 가능 여부와 장애를 확인하며, CPU·메모리 requests와 limits도 정의하겠습니다. 로그와 모니터링을 구성하고, 모